### **ACID - PART 1 - Extract field of view**

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2026/02/06

## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [6]:
# Import required modules
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd

# from scipy.ndimage import median_filter, binary_fill_holes #to check
# from skimage.filters import threshold_otsu #to check
# from skimage.measure import label #to check
# from skimage.transform import resize
from ome_types import to_xml
from acid.utils.listdirNHF import listdirNHF
from acid.utils.mksubdir import mkdir_tree
from acid.image_processing.extract_metadata import extract_bioio_scene_metadata
from acid.image_processing.name_metadata import extract_name_metadata
from acid.image_processing.make_imagej_metadata import imagej_compatible_metadata_dict
from acid.utils.open_image import bioio_open_image
from acid.utils.save_image import tifffile_save_ometiff
from acid.image_processing.save_metadata import save_xml_string

### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [7]:
# indicate the path to the directory storing the input images - NOTE: input images are expected to be
# separated into different sub-directories per each experiment
input_directory = Path.cwd().parent / "data" / "raw"

# indicate the path to the directory where outputs will be saved
output_directory = input_directory / "proc"

# Indicate the markers used for each channel - if possible, try to be the most explicit possible (e.g. indicate dapi instead of nucleus,
# indicate phalloidin instead of cytoskeleton)
channel_0 = "Hoechst"
channel_1 = "Concanavalin-A-488"
channel_2 = "Phalloidin-568"
channel_3 = "anti-Nonstructural-Protein-3-647"
channel_4 = "Differential-Interference-Contrast"


# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

# --- metadata information to be included in the ome.tif files ---
# inticate the unit of the x and y physical size of the images as extracted from the .nd2 raw file metadata
size_unit = "micron"  # should be implemented using image_preparation.extract_metadata.extract_physical_size_unit_from_nd2_xml()

# location
location = "Center for Integrative Infectious Disease Research - Heidelberg (Germany)"

# microscope
microscope = "Nikon Ti2 - CSU-W1 - spinning disc"

# objective
objective = "Nikon Apochromat Lambda-S 60x/1.40 Oil"

# processing_date format
processing_date_format = "%y%m%d"

# --- parameters for working with input file names ---
# indicate the part of the name to use to distinguish files and folders to process or exclude
# among those into the input_directory. NOTE: when this was first implemented the input_directory
# contained sub-directories (experiments) to process and a powerpoint presentation to ignore
input_dir_target = None
input_dir_exclude = [".pptx", ".csv", ".xlsx"]


# Indicate the part of the input file name to use to filter input files to analyse from files to ignore
# The input file contains .nd2 - input_file_exclude set to None indicates that no sub-string should be used to identify
# files in the input directory sub-directories (individual experiments) which should not be processed.
# As a consequence, all files containing the .nd2 string will be processed
input_file_target = ".nd2"
input_file_exclude = None

# experiment sub-directory separator - used to split experiment sub-directory name and extract
# metadata information within it
experiment_separator = "_"

# sub-experiment separator - experiment directory names contain "."
# this will be problematic for name handling in downstream analysis, so it will be replaced with
# the string indicated in variable "sub_exp_sep_replacement" (see below)
sub_exp_sep = "."

# separator - used to split file .nd2 raw file name and extract metadata information within it
file_name_separator = "_"


# --- parameters for extracting sub-info bits from file and experiment names ---
# index of the experiment name in the experiment subdirectory splat using the experiment_separator
experiment_index = -1

# bitinfo tuples to be passed to the extract_name_metadata function (ref to image_preparation.name_metadata.extract_name_metadata
# for details on the structure of these tuples)
condition_1_bitinfo = (0, None, None, None)
infectious_organism_bitinfo = (1, None, None, None)
condition_2_bitinfo = (2, None, None, None)
imaging_hours_bitinfo = (3, None, None, None)
well_bitinfo = (-1, None, None, None)

# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = "_"

# project
project_name = "ACID"

# xml suffix - used to save .xml files
xml_suffix = "str.xml"

# ome suffix - used to save ome.tif files
ome_suffix = ".ome.tif"

# ignore indexes when saving pandas dataframes as csv files
save_csv_index = (
    False  # if False, the index will not be saved as a separate column in the csv file
)

# metadata saving date format
metadata_date_format = "%Y%m%d"

# metadata savingword
metadata_savingword = "metadata"

# metadata file suffix
metadata_file_suffix = f"part{save_file_name_separator}1.csv"

# hyperparameters saving date format
hyperparameters_date_format = "%Y%m%d-%H%M%S"

# hyperparameters savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters file suffix
hyperparameters_file_suffix = f"part{save_file_name_separator}1.csv"

# indicate the name the order of the dimensions will be indicated as in the metadata of the saved ome file
dims_order_name = "dims_order"

# experiment separator replacement - experiment directory names contain "." (see sub_exp_sep above)
# this will be problematic for name handling in downstream analysis, so it will be replaced with
# the string indicated in the variable below
sub_exp_sep_replacement = "p"

# indicate whether ome.tif files should be saved as an imagej compatible format
# NOTE: this is the default behaviour and the recommended one. The result of setting this to False hasn't been tested
save_imagej_compatible = True

# indicate the photometric interpretation to be used when saving the ome.tif files
# NOTE: at the moment, only 'minisblack' has been tested
photometric = "minisblack"


# --- parameters for saving directories ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_name = "secondary_output"
# exist_ok = True # if the secondary output directory already exists, do not raise an error

# indicate the name of the directory for saving metadata on the processed files and processing process
metadata_name = "proc_metadata"

# indicate the name of the directory for saving extracted fields of view
fov_name = "fov"

### Create secondary output directory - this directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [8]:
# # create the path to secondary_output directory
# secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# # create the secondary_output directory
# os.makedirs(secondary_output_path, exist_ok=exist_ok)

secondary_output_path, metadata_path, fov_path = mkdir_tree(
    secondary_output_name=secondary_output_name,
    metadata_name=metadata_name,
    metadata_parent=output_directory,
    fov_name=fov_name,
    fov_parent=output_directory,
)

### MAIN LOOP
#### 1.1. Iterate through experiment folders.
#### 1.2. Iterate through raw files.
#### 1.3. Extract original metadata.
#### 1.4. Save original metadata.
#### 1.5. Extract field of views (also called scenes) to analyse from raw input files.
#### 1.6. Extract metadata from file and scene names
#### 1.7. Save individual fields of view as ome.tif along with their metadata
#### 1.8. Collect field of view -specific original, name and processing metadata into a data frame and save it as csv file.

## **--- --- ---**

Run the following cell.

Don't modify the following cell.

In [9]:
# intialize lists to collect all file metadata
scenes_metadata_collection = []

# iterate through experiment sub-directories in the input directory
for experiment_subdir in listdirNHF(
    Path(input_directory), target=input_dir_target, exclude=input_dir_exclude
):

    print("========= ========= =========")
    print((f"working on experiment sub-directory {experiment_subdir}"))

    # set the full path to the input directory for the current experiment
    input_directory_exp = os.path.join(input_directory, experiment_subdir)

    # Import input file names as a list - don't modify the following line
    input_file_list = listdirNHF(
        Path(input_directory_exp), target=input_file_target, exclude=input_file_exclude
    )

    # iterate through the target files in the input directory
    for input_file in input_file_list:

        print("=========")
        print((f"working on {input_file}"))

        # ---------   ---------
        # READ THE FILE AND THE ORIGINAL METADATA
        # ---------   ---------
        bioio_input_image, input_image_metadata = bioio_open_image(
            os.path.join(input_directory_exp, input_file), return_metadata=True
        )

        # ---------   ---------
        # SAVE THE ORIGINAL METADATA
        # ---------   ---------

        # convert input_image_metadata to xml object
        xml_input_image_metadata = to_xml(input_image_metadata)

        # save file metadata in the fieals of view directory
        save_xml_string(
            xml_input_image_metadata,
            os.path.join(
                fov_path,
                f"{input_file.removesuffix(input_file_target)}{save_file_name_separator}{xml_suffix}",
            ),
        )

        # ---------   ---------
        # ITERATE THROUGH THE SCENES WITHIN THE FILE
        # ---------   ---------
        # iterate through the scenes
        for scene_n, scene in enumerate(bioio_input_image.scenes):

            print("---------")
            print(f"working on scene {scene}")

            # set scene
            bioio_input_image.set_scene(scene)

            # ---------   ---------
            # EXTRACT METADATA FROM FILE NAME
            # ---------   ---------

            # extract experiment number form experiment sub-directory name
            experiment = experiment_subdir.split(experiment_separator)[experiment_index]

            # extract metadata from file name
            name_metadata_dict = extract_name_metadata(
                input_file.removesuffix(input_file_target),
                separator=file_name_separator,
                infobits={
                    "processing_date": datetime.datetime.now().strftime(
                        processing_date_format
                    ),
                    "experiment": experiment,
                    "condition_1": condition_1_bitinfo,
                    "infectious_organism": infectious_organism_bitinfo,
                    "condition_2": condition_2_bitinfo,
                    "imaging_hours": imaging_hours_bitinfo,
                    "well": well_bitinfo,
                },
            )
            # form the saving name of the file
            save_file_name = f"{input_file.removesuffix(input_file_target)}{file_name_separator}{experiment.replace(sub_exp_sep, sub_exp_sep_replacement)}{file_name_separator}{scene}{ome_suffix}"

            # extract metadata from raw image metadata, add extra info, return metadata in their final version
            scene_metadata_series, scene_metadata_dict = extract_bioio_scene_metadata(
                bioio_scene=bioio_input_image,
                dims_order_name=dims_order_name,
                raw_file_name=input_file,
                scene_name=scene,
                processing_date_yymmdd=name_metadata_dict["processing_date"],
                ome_tif_file_name=save_file_name,
                location=location,
                microscope=microscope,
                objective=objective,
                experiment=experiment,
                condition_1=name_metadata_dict["condition_1"],
                infectious_organism=name_metadata_dict["infectious_organism"],
                condition_2=name_metadata_dict["condition_2"],
                imaging_hours=name_metadata_dict["imaging_hours"],
                well=name_metadata_dict["well"],
                channel_0=channel_0,
                channel_1=channel_1,
                channel_2=channel_2,
                channel_3=channel_3,
                channel_4=channel_4,
                physical_size_unit_x=size_unit,
                physical_size_unit_y=size_unit,
            )
            # ---------   ---------
            # COLLECT SCENE METADATA IN THE COLLECTION LIST
            # ---------   ---------

            # append scene_metadata_series to collection list
            scenes_metadata_collection.append(scene_metadata_series)

            # ---------   ---------
            # SAVE OME.TIF FILE
            # ---------   ---------
            # get data as an array
            input_scene = bioio_input_image.data

            # Remove axis of size 1 - this will make the image compatible with ImageJ
            input_scene = np.squeeze(input_scene)

            # add "custom_" to each metadata contained in metadata dictionary, so that they can be
            # recognized when opened with ImageJ
            imagej_scene_metadata_dict = imagej_compatible_metadata_dict(
                scene_metadata_dict
            )

            tifffile_save_ometiff(
                os.path.join(fov_path, save_file_name),
                data=input_scene,
                imagej=save_imagej_compatible,
                photometric=photometric,
                metadata=imagej_scene_metadata_dict,
            )

# ---------   ---------
# FORM A DATA FRAME WITH ALL METADATA
# ---------   ---------
# concatenate scenes metadata into a pandas data frame
metadata_df = pd.concat(scenes_metadata_collection, axis=1).T


# save metadata dataframe as a csv file
metadata_saving_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
metadata_df.to_csv(
    os.path.join(metadata_path, metadata_saving_name), index=save_csv_index
)

print("")
print("Finished")

========= ========= =========
working on experiment sub-directory experiment_A07.4
working on H7_DENV2_MOI1_40h_fixed_stained_well6.nd2
---------
working on scene A1
---------
working on scene A2
---------
working on scene A3
---------
working on scene A4
---------
working on scene A5
---------
working on scene A6
---------
working on scene A7
---------
working on scene B7
---------
working on scene B6
---------
working on scene B5
---------
working on scene B4
---------
working on scene B3
---------
working on scene B2
---------
working on scene B1
---------
working on scene C1
---------
working on scene C2
---------
working on scene C3
---------
working on scene C4
---------
working on scene C5
---------
working on scene C6
---------
working on scene C7
---------
working on scene D7
---------
working on scene D6
---------
working on scene D5
---------
working on scene D4
---------
working on scene D3
---------
working on scene D2
---------
working on scene D1
---------
working on sce

### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [10]:
# # collect hyperparameters in a dictionary

hyperparameter_dict = {
    "input_directory": input_directory,
    "output_directory": output_directory,
    "channel_0": channel_0,
    "channel_1": channel_1,
    "channel_2": channel_2,
    "channel_3": channel_3,
    "channel_4": channel_4,
    "size_unit": size_unit,
    "location": location,
    "microscope": microscope,
    "objective": objective,
    "input_dir_target": input_dir_target,
    "input_dir_exclude": input_dir_exclude,
    "input_file_target": input_file_target,
    "input_file_exclude": input_file_exclude,
    "experiment_separator": experiment_separator,
    "sub_exp_sep": sub_exp_sep,
    "file_name_separator": file_name_separator,
    "experiment_index": experiment_index,
    "condition_1_bitinfo": condition_1_bitinfo,
    "infectious_organism_bitinfo": infectious_organism_bitinfo,
    "condition_2_bitinfo": condition_2_bitinfo,
    "imaging_hours_bitinfo": imaging_hours_bitinfo,
    "well_bitinfo": well_bitinfo,
    "save_file_name_separator": save_file_name_separator,
    "project_name": project_name,
    "xml_suffix": xml_suffix,
    "ome_suffix": ome_suffix,
    "save_csv_index": save_csv_index,
    "metadata_date_format": metadata_date_format,
    "metadata_savingword": metadata_savingword,
    "metadata_file_suffix": metadata_file_suffix,
    "hyperparameters_date_format": hyperparameters_date_format,
    "hyperparameters_savingword": hyperparameters_savingword,
    "hyperparameters_file_suffix": hyperparameters_file_suffix,
    "dims_order_name": dims_order_name,
    "sub_exp_sep_replacement": sub_exp_sep_replacement,
    "save_imagej_compatible": save_imagej_compatible,
    "photometric": photometric,
    "secondary_output_name": secondary_output_name,
    "metadata_name": metadata_name,
    "fov_name": fov_name,
}


# transform the hyperparameter_dict in a pandas series
hyperparameter_series = pd.Series(hyperparameter_dict)

# save hyperparamters
hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{hyperparameters_savingword}{save_file_name_separator}{hyperparameters_file_suffix}"
hyperparameter_series.to_csv(
    os.path.join(secondary_output_path, hyperparameter_saving_name),
    index=save_csv_index,
)